In [1]:
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms

In [2]:
# try running the following code, if you see errors on Windows OS

# import os
# os.environ['KMP_DUPLICATE_LIB_OK']='TRUE'

In [3]:
# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [4]:
# Hyper-parameters 
input_size = 784
hidden_size = 500
num_classes = 10
num_epochs = 5
batch_size = 100
learning_rate = 0.001

In [5]:
# MNIST dataset 
train_dataset = torchvision.datasets.MNIST(root='data', 
                                           train=True, 
                                           download=True,
                                           transform=transforms.ToTensor())

test_dataset = torchvision.datasets.MNIST(root='data', 
                                          train=False, 
                                          transform=transforms.ToTensor(),
                                          download=True)

# Data loader
train_loader = torch.utils.data.DataLoader(dataset=train_dataset, 
                                           batch_size=batch_size, 
                                           shuffle=True)

test_loader = torch.utils.data.DataLoader(dataset=test_dataset, 
                                          batch_size=batch_size, 
                                          shuffle=False)

In [6]:
len(train_dataset),len(test_dataset)

(60000, 10000)

In [7]:
# Fully connected neural network with one hidden layer
class NeuralNet(nn.Module):
    def __init__(self, input_size, hidden_size, num_classes):
        super(NeuralNet, self).__init__()
        self.fc1 = nn.Linear(input_size, hidden_size) 
        self.sigmoid = nn.Sigmoid()
        self.fc2 = nn.Linear(hidden_size, num_classes)  
    
    def forward(self, x):
        out = self.fc1(x)
        out = self.sigmoid(out)
        out = self.fc2(out)
        return out

model = NeuralNet(input_size, hidden_size, num_classes).to(device)

In [8]:
# Loss and optimizer
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)  

In [9]:
# Train the model
total_step = len(train_loader)
for epoch in range(num_epochs):
    for i, (images, labels) in enumerate(train_loader):  
        # Move tensors to the configured device
        images = images.reshape(-1, 28*28).to(device)
        labels = torch.nn.functional.one_hot(labels,num_classes=10).float().to(device)
        
        # Forward pass
        outputs = model(images)
        loss = criterion(outputs, labels)
        
        # Backward and optimize
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        if (i+1) % 100 == 0:
            print ('Epoch [{}/{}], Step [{}/{}], Loss: {:.4f}' 
                   .format(epoch+1, num_epochs, i+1, total_step, loss.item()))


Epoch [1/5], Step [100/600], Loss: 0.0479
Epoch [1/5], Step [200/600], Loss: 0.0348
Epoch [1/5], Step [300/600], Loss: 0.0404
Epoch [1/5], Step [400/600], Loss: 0.0316
Epoch [1/5], Step [500/600], Loss: 0.0312
Epoch [1/5], Step [600/600], Loss: 0.0290
Epoch [2/5], Step [100/600], Loss: 0.0221
Epoch [2/5], Step [200/600], Loss: 0.0217
Epoch [2/5], Step [300/600], Loss: 0.0217
Epoch [2/5], Step [400/600], Loss: 0.0183
Epoch [2/5], Step [500/600], Loss: 0.0195
Epoch [2/5], Step [600/600], Loss: 0.0146
Epoch [3/5], Step [100/600], Loss: 0.0195
Epoch [3/5], Step [200/600], Loss: 0.0145
Epoch [3/5], Step [300/600], Loss: 0.0117
Epoch [3/5], Step [400/600], Loss: 0.0104
Epoch [3/5], Step [500/600], Loss: 0.0118
Epoch [3/5], Step [600/600], Loss: 0.0109
Epoch [4/5], Step [100/600], Loss: 0.0121
Epoch [4/5], Step [200/600], Loss: 0.0086
Epoch [4/5], Step [300/600], Loss: 0.0106
Epoch [4/5], Step [400/600], Loss: 0.0106
Epoch [4/5], Step [500/600], Loss: 0.0131
Epoch [4/5], Step [600/600], Loss:

In [10]:
# Test the model
# In test phase, we don't need to compute gradients (for memory efficiency)
with torch.no_grad():
    correct = 0
    total = 0
    for images, labels in test_loader:
        images = images.reshape(-1, 28*28).to(device)
        labels = labels.to(device)
        outputs = model(images)
        # _, predicted = torch.max(outputs.data, 1)
        predicted = torch.argmax(outputs.data,dim=1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    print('Accuracy of the network on the 10000 test images: {} %'.format(100 * correct / total))

Accuracy of the network on the 10000 test images: 96.09 %
